## https://github.com/2Myaka2/DataScience_projects

### работал в репозитории

# **Нелинейные модели против южной погоды**


## Описание проекта

Компания **BikeSouth** развивает городской велопрокат на юге России и хочет улучшить систему прогнозирования спроса на аренду велосипедов. Для бизнеса важно заранее понимать, сколько велосипедов потребуется в каждый конкретный час, чтобы эффективнее распределять велосипеды между станциями, снижать количество простоев и избегать ситуаций, когда клиент не может арендовать велосипед из-за дефицита.

Ранее компания использовала базовую модель на основе **линейной регрессии**, однако её качество оказалось недостаточным. Вероятная причина заключается в том, что спрос на велосипеды зависит не только от отдельных факторов, но и от их сочетаний: температуры, влажности, осадков, солнечной активности, времени суток, сезона и праздничных дней. Такие зависимости могут быть нелинейными, поэтому в проекте будут рассмотрены более гибкие модели машинного обучения.

## Цель проекта

Разработать модель машинного обучения, которая предсказывает значение целевой переменной:

**`Rented Bike Count`** — количество арендованных велосипедов за конкретный час.

Модель должна учитывать погодные и календарные признаки и показывать качество выше, чем предоставленная baseline-модель на основе линейной регрессии.

## Задачи проекта

В рамках проекта необходимо:

1. Загрузить тренировочную и тестовую выборки.
2. Загрузить предоставленный baseline-пайплайн с линейной регрессией из `.pkl`-файла.
3. Оценить качество baseline-модели на train и test по метрикам:
   - RMSE;
   - MAE;
   - R².
4. Провести первичный анализ данных:
   - проверить типы признаков;
   - изучить пропуски;
   - оценить уникальные значения;
   - проверить распределение целевой переменной.
5. Выполнить EDA и сформулировать гипотезы о факторах, влияющих на спрос.
6. Подготовить данные для обучения моделей без утечки данных.
7. Обучить две нелинейные модели:
   - `KNeighborsRegressor`;
   - `DecisionTreeRegressor`.
8. Подобрать гиперпараметры для обеих моделей с помощью `Optuna`.
9. Использовать единую схему кросс-валидации с 5 фолдами.
10. Сравнить лучшие версии моделей по метрикам RMSE, MAE и R².
11. Выбрать лучшую модель и один раз проверить её на тестовой выборке.
12. Сравнить итоговое качество лучшей модели с baseline-моделью.
13. Проанализировать важность признаков и сделать выводы для бизнеса.

## Особенности валидации

Тестовая выборка используется только для финальной оценки качества модели.  
Она не применяется при обучении, подборе гиперпараметров и выборе модели.

Для подбора гиперпараметров и сравнения моделей используется кросс-валидация на тренировочной выборке.

Основная метрика оптимизации:

**RMSE** — корень из среднеквадратичной ошибки.

Чем ниже RMSE и MAE, тем лучше модель.  
Чем выше R², тем лучше модель объясняет вариативность целевой переменной.

## Используемые модели

В проекте будут обучены и сравнены следующие модели:

- **Baseline Linear Regression** — предоставленная модель компании BikeSouth.
- **kNN Regressor** — модель на основе ближайших соседей.
- **Decision Tree Regressor** — дерево решений для задачи регрессии.

## Ожидаемый результат

Итогом проекта должна стать модель, которая лучше baseline-модели прогнозирует почасовой спрос на аренду велосипедов. Полученные результаты могут помочь BikeSouth:

- точнее планировать количество доступных велосипедов;
- улучшать логистику перераспределения велосипедов;
- снижать риск дефицита велосипедов в часы повышенного спроса;
- повышать качество клиентского опыта.

In [2]:
%%writefile requirements.txt
numpy==1.26.4
pandas==2.2.3
scipy==1.13.1
scikit-learn==1.6.1
matplotlib==3.10.8
seaborn==0.13.2
joblib==1.5.3
ipykernel==6.29.5
optuna==4.4.0

Overwriting requirements.txt


In [3]:
# Базовые библиотеки
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

# Визуализация
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn: базовые инструменты
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_val_score,
    cross_validate
)

# sklearn: предобработка
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# sklearn: модели
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text

# sklearn: метрики
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
    r2_score,
    make_scorer
)

# sklearn: интерпретация
from sklearn.inspection import permutation_importance

# Optuna для подбора гиперпараметров
import optuna

# Настройки отображения
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.3f}".format)

sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
CV_FOLDS = 5

/home/andre/DS/DataScience_projects/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Загрузка данных

REMOTE_DATA_URL = "https://code.s3.yandex.net/datasets/"

file_names = {
    "train": "ds_s14_train_data.csv",
    "test": "ds_s14_test_data.csv",
}

dataframes = {}

for name, file_name in file_names.items():
    local_path = Path("datasets") / file_name          # локальная папка проекта
    practicum_path = Path("/datasets") / file_name     # путь на платформе Практикума
    remote_path = REMOTE_DATA_URL + file_name          # удалённый путь
    
    try:
        dataframes[name] = pd.read_csv(local_path)
        print(f"{name}: загружен из локальной папки {local_path}")
    except FileNotFoundError:
        try:
            dataframes[name] = pd.read_csv(practicum_path)
            print(f"{name}: загружен из папки Практикума {practicum_path}")
        except FileNotFoundError:
            dataframes[name] = pd.read_csv(remote_path)
            print(f"{name}: загружен из удалённого хранилища {remote_path}")

df_train = dataframes["train"]
df_test = dataframes["test"]

print()
print(f"Размер тренировочной выборки: {df_train.shape}")
print(f"Размер тестовой выборки: {df_test.shape}")

train: загружен из локальной папки datasets/ds_s14_train_data.csv
test: загружен из локальной папки datasets/ds_s14_test_data.csv

Размер тренировочной выборки: (7008, 16)
Размер тестовой выборки: (1752, 16)


In [6]:
df_mains = {
    'df_train': df_train,
    'df_test': df_test
}

In [7]:
def show_df_info(name, df):
    print('=' * 80)
    print(f'Датасет: {name}')
    print('=' * 80)
    
    print(f'Размер: {df.shape[0]} строк, {df.shape[1]} столбцов')
    
    print('\nИнформация о типах данных:')
    df.info()
    
    print('\nПервые 5 строк:')
    display(df.head())
    
    print('\nКоличество пропусков:')
    display(df.isna().sum().to_frame(name='missing_values'))
    
    print('\nКоличество явных дубликатов:')
    print(df.duplicated().sum())
    
    print('\n')

for name, df in df_mains.items():
    show_df_info(name, df)

Датасет: df_train
Размер: 7008 строк, 16 столбцов

Информация о типах данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7008 entries, 0 to 7007
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Temperature               7008 non-null   float64
 1   Humidity(%)               6758 non-null   float64
 2   Wind speed (m/s)          6798 non-null   float64
 3   Visibility (10m)          6749 non-null   float64
 4   Dew point temperature     7008 non-null   float64
 5   Solar Radiation (MJ/m2)   6798 non-null   float64
 6   Rainfall(mm)              6746 non-null   float64
 7   Snowfall (cm)             6745 non-null   float64
 8   Seasons                   7008 non-null   object 
 9   Holiday                   7008 non-null   object 
 10  Functioning Day           7008 non-null   object 
 11  Time_Period_Evening       7008 non-null   bool   
 12  Time_Period_Late Evening  7008 non-null 

,Temperature,Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature,Solar Radiation (MJ/m2),Rainfall(mm),Snowfall (cm),Seasons,Holiday,Functioning Day,Time_Period_Evening,Time_Period_Late Evening,Time_Period_Morning,Time_Period_Night,Rented Bike Count
0,20.300,35.000,2.400,2000.000,4.300,0.460,0.000,0.000,Autumn,Holiday,Yes,True,False,False,False,1237
1,25.400,55.000,3.200,2000.000,15.600,0.150,0.000,0.000,Autumn,No Holiday,Yes,True,False,False,False,2468
2,-6.900,39.000,1.600,2000.000,-18.500,0.000,0.000,0.000,Winter,No Holiday,Yes,False,True,False,False,186
3,-5.200,37.000,2.200,2000.000,-17.600,0.000,0.000,0.000,Winter,No Holiday,Yes,False,False,False,True,254
4,23.400,34.000,2.100,2000.000,6.600,2.840,0.000,0.000,Autumn,No Holiday,Yes,False,False,False,False,1686



Количество пропусков:


,missing_values
Temperature,0
Humidity(%),250
Wind speed (m/s),210
Visibility (10m),259
Dew point temperature,0
Solar Radiation (MJ/m2),210
Rainfall(mm),262
Snowfall (cm),263
Seasons,0
Holiday,0



Количество явных дубликатов:
0


Датасет: df_test
Размер: 1752 строк, 16 столбцов

Информация о типах данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1752 entries, 0 to 1751
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Temperature               1752 non-null   float64
 1   Humidity(%)               1680 non-null   float64
 2   Wind speed (m/s)          1700 non-null   float64
 3   Visibility (10m)          1680 non-null   float64
 4   Dew point temperature     1752 non-null   float64
 5   Solar Radiation (MJ/m2)   1700 non-null   float64
 6   Rainfall(mm)              1688 non-null   float64
 7   Snowfall (cm)             1691 non-null   float64
 8   Seasons                   1752 non-null   object 
 9   Holiday                   1752 non-null   object 
 10  Functioning Day           1752 non-null   object 
 11  Time_Period_Evening       1752 non-null   bool   
 12  Time_Pe

,Temperature,Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature,Solar Radiation (MJ/m2),Rainfall(mm),Snowfall (cm),Seasons,Holiday,Functioning Day,Time_Period_Evening,Time_Period_Late Evening,Time_Period_Morning,Time_Period_Night,Rented Bike Count
0,6.700,48.000,0.800,817.000,-3.500,0.390,0.000,0.000,Winter,No Holiday,Yes,False,False,False,False,402
1,-5.300,90.000,1.100,311.000,-6.600,0.000,0.000,2.200,Winter,No Holiday,Yes,False,False,True,False,178
2,2.900,66.000,1.200,173.000,-2.800,0.000,0.000,0.000,Winter,No Holiday,Yes,False,True,False,False,219
3,23.100,63.000,2.600,949.000,15.600,1.680,0.000,0.000,Spring,No Holiday,Yes,False,False,False,False,949
4,23.300,43.000,1.200,1257.000,10.000,1.910,0.000,0.000,Summer,No Holiday,Yes,False,False,True,False,1005



Количество пропусков:


,missing_values
Temperature,0
Humidity(%),72
Wind speed (m/s),52
Visibility (10m),72
Dew point temperature,0
Solar Radiation (MJ/m2),52
Rainfall(mm),64
Snowfall (cm),61
Seasons,0
Holiday,0



Количество явных дубликатов:
0




## **Часть 1. Работа с базовой моделью**

Сперва вы познакомитесь с тем, как работает baseline-модель, которую использовала компания BikeSouth до того, как обратилась к вам.

Компания предоставила:

* Pickle-файл — готовый обученный пайплайн без исходного кода. Доступен по пути здесь: `'/datasets/baseline_linear_regression_pipeline.pkl'`.

* Тренировочную и тестовую выборки, которые можно использовать для оценки модели. Они расположены здесь:

  * `'/datasets/ds_s14_train_data.csv'`;
  * `'/datasets/ds_s14_test_data.csv'`.


Базовую модель не нужно обучать заново — достаточно загрузить её и проверить качество.

**Совет:**
1. Убедитесь, что у вас есть доступ к `baseline_linear_regression_pipeline.pkl`, `ds_s14_train_data.csv` и `ds_s14_test_data.csv`.

2. Разделите тестовый набор на признаки `X` и целевую переменную `y`.

3. Загрузите .pkl-файл — это готовый пайплайн, который сам обрабатывает данные и делает предсказания. Модель автоматически применяет трансформации и возвращает прогнозы.

4. Посчитайте RMSE, MAE и R² — эти результаты нужны для оценки ваших улучшенных моделей в дальнейшей работе.

>Техническая напоминание: для работы с PKL-файлом нужно установить библиотеку joblib.

In [ ]:
#!pip install "scikit-learn==1.6.1" "numpy==1.26.4"

## **Часть 2. Улучшение модели — kNN и дерево решений**

Ваша задача — предложить более гибкую модель прогноза спроса на велосипеды, которая учитывает нюансы погоды и поведение клиентов.

Вы будете экспериментировать с моделями kNN и деревьями решений, используя подбор гиперпараметров Optuna.

**Шаг 1. Изучение данных**

Проведите исследовательский анализ данных:
1. Посмотрите на распределение целевой переменной `Rented Bike Count`. Определите, есть ли у неё выбросы или сильные сезонные колебания.
2. Постройте графики зависимости спроса от разных признаков.
3. Сравните спрос в разные сезоны и праздничные дни.
4. Рассчитайте корреляцию между признаками и целевой переменной.

**Совет:**

Не нужно сразу всё усложнять: начните с базовых графиков и описательной статистики.



---



**Шаг 2. Разделение данных на тренировочную и валидационную выборки**

Используйте на этом этапе данные файла `ds_s14_train_data.csv`. Тестовый набор нужен только для финальной оценки модели после обучения и подбора гиперпараметров.

Подготовка данных вам понадобится, чтобы обучить модель и оценить её качество через валидацию.

---



**Шаг 3. Обучение новых моделей**

kNN и деревья решений могут уловить нелинейные зависимости, недоступные линейной регрессии. Пора это проверить!

1. Подготовьте пайплайн для каждой модели:
    * Выполните предобработку данных.
    * Инициализизируйте регрессионные модели kNN и дерево решений.
4. Настройте базовые параметры моделей — например, `n_neighbors` для kNN, `max_depth` для дерева.

**Совет:**

Начинайте с базовых параметров, чтобы убедиться, что пайплайн работает. Оптимизацию параметров вы сделаете на следующем шаге.

---



**Шаг 4. Подбор гиперпараметров с Optuna**

Компания хочет точную модель. Optuna поможет найти лучшие гиперпараметры для kNN и дерева, чтобы снизить ошибки прогноза.


1. Определите функцию цели для Optuna.

2. Настройте диапазоны гиперпараметров.

3. Запустите оптимизацию и сохраните лучшие параметры.

**Совет:**

Не бойтесь сначала экспериментировать с небольшими диапазонами, а потом расширять их, если модель не уловит зависимости.

---



**Шаг 5. Кросс-валидация новых моделей**

1. Проведите кросс-валидацию kNN и дерева решений с оптимальными гиперпараметрами.
2. Сравните метрики с baseline-моделью.
3. Определите, какая модель показывает лучшие результаты на тренировочной выборке.

**Совет:**

Используйте визуализации (например, столбчатую диаграмму или ящик с усами), чтобы оценить разброс метрик и стабильность моделей.


---



**Шаг 6. Составление отчёта по моделям**

1. Составьте таблицу с метриками для трёх моделей: baseline, лучшей kNN и лучшего дерева решений.
2. Добавьте визуализацию с распределением метрик, если необходимо.

Подготовьте выводы:
* Какая модель лучше справляется с прогнозом?
* Какие признаки, по вашему мнению, особенно важны?


**Совет:**

Старайтесь объяснить результаты в бизнес-контексте. Примеры выводов на языке заказчика:
* «Эта модель лучше реагирует на дождь».
* «Температура и влажность сильно влияют на спрос в пиковые часы».

---



**Шаг 7. Сохранение модели и отчёта**

1. Выберите финальную, лучшую модель и оцените её качество на тестовой выборке, чтобы понять, насколько хорошо она прогнозирует на новых данных.
2. Подготовьте тетрадку с кодом и комментариями: включите результаты всех экспериментов, метрики моделей, визуализации, а также обоснование выбора финальной модели.

**Совет:**

Документируйте каждый шаг. Объясняйте, почему выбраны те или иные гиперпараметры и подходы. В реальной бизнес-задаче эта привычка поможет вашим коллегам и руководству понимать решения и доверять модели.

---



**Дополнительное задание: реализация кастомного трансформера**

Простые признаки вроде температуры или влажности не всегда отражают реальную ситуацию. Чтобы модель могла лучше прогнозировать спрос на велосипеды, можно создавать новые признаки, которые учитывают особенности погоды или взаимодействия факторов.

1. Реализуйте класс с методами `fit` и `transform`.
2. Вставьте его в пайплайн перед моделью.
3. Убедитесь, что трансформер корректно работает с тренировочными данными.

**Совет:**  
Если вы решите реализовать трансформер, начинайте с простых комбинаций признаков, чтобы не усложнять модель слишком рано.


---

